# EMERALD — Traffic Violation Detection Pipeline

**Version:** 8.1  
**Framework:** YOLOv8 · Supervision · ByteTrack · EasyOCR  
**Target environment:** Google Colab (GPU) or local Jupyter with CUDA

---

## Abstract

EMERALD is a computer-vision pipeline for automated detection and logging of traffic violations from video footage. The system processes video frame-by-frame, applying object detection, multi-object tracking, and a rule-based violation engine to produce time-stamped, evidence-annotated records for each infraction. The following violation categories are supported: no-helmet riding, triple riding, red-light crossing, illegal parking, and wrong-side driving.

---

## Version History

| Version | Change Summary |
|---------|---------------|
| v8.1 | Introduced dual-tracker architecture (`moto_tracker` / `lv_tracker`) to resolve motorcycle tracking loss caused by uniform ByteTrack parameters. Large-vehicle track IDs offset by 50,000 to prevent ID collision between independent tracker counters. |
| v8 | Raised per-class confidence floors for cars and large vehicles to suppress road-paint false positives. Added minimum bounding-box area and aspect-ratio filters for large-vehicle detections. Implemented stale-track pruning in `Tracker.update()` and ghost-track suppression in `draw_dashboard_overlay`. |
| v3 | Fixed confidence-floor regression for COCO classes 2/5/7; corrected fallback merge bug that discarded large-vehicle detections when two-wheeler fallback was active. Added `DEBUG_DETECTOR` toggle. |
| v3 (base) | Upgraded to YOLOv8s backbone; added multi-frame helmet voting, per-class adaptive confidence floors, and motorcycle minimum-area filter. |


## Pipeline Architecture

The pipeline consists of eight sequential stages:

1. **Environment setup** — dependency installation, GPU verification, global parameter configuration
2. **Scene calibration** — lane polygon definitions, no-parking zones, stop-line regions
3. **Frame ingestion** — video decoding, frame sampling, optional CLAHE/unsharp-mask preprocessing
4. **Object detection** — YOLOv8n inference with per-class confidence floors and shape-based false-positive rejection
5. **Helmet detection** — dedicated two-wheeler head-crop extraction and multi-frame voting via a secondary YOLO model
6. **Multi-object tracking** — dual ByteTrack instances with class-group-specific parameters; track-state accumulation
7. **Violation engine** — geometry-based rule evaluation (helmet, red-light, parking, wrong-side, triple-riding)
8. **OCR and evidence storage** — EasyOCR license-plate reading; annotated frame export and JSONL log


## 1. Environment Setup


In [1]:
import torch

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")


CUDA Available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [ ]:
!pip install ultralytics supervision easyocr ipywidgets --quiet
import supervision as sv
print(f"Supervision version: {sv.__version__}")


## 2. Configuration

All tunable parameters are defined in this section. Modify values here rather than inside class or function bodies.

**Tracker parameters** are split into two groups reflecting the dual-tracker design:
- `MOTO_*` — applied to persons, bicycles, and motorcycles (permissive settings).
- `LV_*` — applied to cars, buses, and trucks (strict settings to suppress road-paint false positives).

`LV_TRACK_ID_OFFSET` ensures that the independent ID counters of the two ByteTrack instances never produce colliding track IDs.


In [3]:
import cv2
import numpy as np
import json, os, time, re
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from collections import defaultdict, deque

import supervision as sv
from ultralytics import YOLO
import easyocr

print("All imports OK")

# Output directories
OUTPUT_DIR = Path("violation_output")
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "frames").mkdir(exist_ok=True)

# ── Frame sampling ─────────────────────────────────────────────────────────
FRAME_SAMPLE_RATE       = 2        # process every Nth decoded frame
ADAPTIVE_SKIP           = True

# ── Detection model ────────────────────────────────────────────────────────
MAIN_MODEL_NAME         = "yolov8n.pt"

# Global minimum confidence for inference; per-class floors below take precedence.
YOLO_CONF_THRESHOLD     = 0.25
YOLO_IMGSZ              = 1920

# Per-class confidence floors applied after inference.
# Car/bus/truck floors are raised above the baseline to reject road-paint detections.
CLASS_CONF_FLOORS       = {
    0:  0.30,   # person
    1:  0.18,   # bicycle
    2:  0.35,   # car    (raised 0.25→0.35 to suppress road-marking false positives)
    3:  0.18,   # motorcycle
    5:  0.30,   # bus    (raised 0.25→0.30)
    7:  0.30,   # truck  (raised 0.25→0.30)
    9:  0.28,   # traffic light
}

# Minimum bounding-box area as a fraction of total frame area for motorcycles.
MOTO_MIN_BOX_AREA_FRAC  = 0.0004

# ── Large-vehicle shape filters ────────────────────────────────────────────
# Road-paint markings are either too small in area or have aspect ratios
# outside the range expected for any real vehicle in a traffic-camera view.
CAR_MIN_BOX_AREA_FRAC  = 0.0006   # approximately 51×51 px at 1920-wide frame
CAR_ASPECT_RATIO_MIN   = 0.25     # w/h lower bound
CAR_ASPECT_RATIO_MAX   = 5.00     # w/h upper bound

# ── Tracker staleness thresholds ───────────────────────────────────────────
TRACK_MAX_AGE_FRAMES   = 30       # frames before a TrackState entry is purged
TRACK_DISPLAY_AGE      = 15       # frames before an on-screen box is hidden

# ── Two-wheeler / person ByteTrack settings (permissive) ──────────────────
# Motorcycles are detected at low confidence and may be occluded intermittently.
MOTO_TRACK_ACTIVATION_THRESH = 0.25   # minimum detection confidence to initialise a track
MOTO_LOST_TRACK_BUFFER       = 20     # frames a lost track is retained before deletion
MOTO_MIN_CONSECUTIVE_FRAMES  = 1      # single-frame confirmation is sufficient

# ── Large-vehicle ByteTrack settings (strict) ─────────────────────────────
# Road-paint ghost detections rarely persist across two consecutive frames,
# so requiring 2-frame confirmation eliminates them without degrading recall.
LV_TRACK_ACTIVATION_THRESH   = 0.40
LV_LOST_TRACK_BUFFER         = 15     # approximately 0.6 s at 25 fps
LV_MIN_CONSECUTIVE_FRAMES    = 2

# ID offset applied to large-vehicle track IDs to prevent collision with
# motorcycle/person IDs when both ByteTrack instances count from 1.
LV_TRACK_ID_OFFSET           = 50_000

# ── Helmet detection ───────────────────────────────────────────────────────
HELMET_CONF             = 0.20
HELMET_VOTE_WINDOW      = 5        # rolling window length (frames)
HELMET_VOTE_THRESH      = 0.40     # minimum fraction of positive frames to confirm helmet
HELMET_PAD_TOP_FRAC     = 0.90     # upward extension above vehicle box top as a fraction of box height
HELMET_PAD_SIDE_FRAC    = 0.20     # lateral extension as a fraction of box width

# ── Violation confirmation ─────────────────────────────────────────────────
MIN_CONFIRM_FRAMES      = 3        # minimum track length before violation rules are evaluated
PARKING_STILL_SECS      = 5        # stationary duration threshold for illegal-parking detection
PIXEL_MOVE_THRESH       = 8        # displacement in pixels below which a vehicle is considered stationary

# ── Wrong-side voting ──────────────────────────────────────────────────────
WRONG_SIDE_VOTE_WINDOW  = 5        # direction history window length
WRONG_SIDE_VOTE_THRESH  = 0.6      # fraction of readings that must agree before flagging

VIDEO_SOURCE            = 0

# ── Debug flags ────────────────────────────────────────────────────────────
DEBUG_DETECTOR          = False    # print per-class detection counts at each debug interval
DEBUG_EVERY_N_FRAMES    = 10

print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Main model: {MAIN_MODEL_NAME}  |  imgsz={YOLO_IMGSZ}")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
All imports OK
Output  -> /content/violation_output
Main model: yolov8n.pt  |  imgsz=1920


In [4]:
# ── Violation rule toggles ─────────────────────────────────────────────────
# Set individual rules to False to disable them without modifying the engine.
ENABLE_NO_PARKING    = True
ENABLE_STOP_LINE     = True
ENABLE_HELMET        = True
ENABLE_WRONG_SIDE    = True
ENABLE_TRIPLE_RIDING = True

# When True, skips CLAHE/unsharp-mask preprocessing to reduce per-frame latency.
# Set False for recorded video to enable full contrast enhancement.
LIVE_FEED_MODE       = False


## 3. Video Input

The pipeline accepts video input via a local file-upload mechanism, removing any dependency on Google Drive or hardcoded paths. The upload cell detects the runtime environment and uses the appropriate method:

- **Google Colab:** `google.colab.files.upload()` opens a native browser file picker.
- **Jupyter / JupyterLab:** an `ipywidgets.FileUpload` widget renders an inline upload button.

Supported formats: `.mp4`, `.avi`, `.mov`, `.mkv`.

After upload, `VIDEO_PATH` is set to the local path of the saved file. Run the validation cell immediately after to confirm the path is set before proceeding.


In [ ]:
import os, shutil
from pathlib import Path

UPLOAD_DIR = Path("uploaded_video")
UPLOAD_DIR.mkdir(exist_ok=True)

VIDEO_PATH = None

def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if _in_colab():
    from google.colab import files as _colab_files

    print("Click 'Choose Files' below and select your traffic video.")
    _uploaded = _colab_files.upload()
    if _uploaded:
        _fname = next(iter(_uploaded))
        VIDEO_PATH = str(UPLOAD_DIR / _fname)
        with open(VIDEO_PATH, "wb") as f:
            f.write(_uploaded[_fname])
        print(f"\nSaved to: {VIDEO_PATH}")
        print("File exists:", os.path.exists(VIDEO_PATH))
    else:
        print("No file was uploaded.")

else:
    try:
        import ipywidgets as widgets
        from IPython.display import display
    except ImportError:
        raise RuntimeError(
            "ipywidgets is required for the upload widget in Jupyter. "
            "Install it with: pip install ipywidgets"
        )

    _upload_widget = widgets.FileUpload(
        accept=".mp4,.avi,.mov,.mkv",
        multiple=False,
        description="Upload video",
    )
    _status_label = widgets.Label(value="No file selected yet.")

    def _on_upload_change(change):
        global VIDEO_PATH
        new_files = change["new"]
        if not new_files:
            return
        # ipywidgets >=8 returns a tuple of dicts; older versions return a dict
        item = new_files[0] if isinstance(new_files, (list, tuple)) else list(new_files.values())[0]
        fname = item["name"]
        content = item["content"]
        dest = UPLOAD_DIR / fname
        with open(dest, "wb") as f:
            f.write(bytes(content))
        VIDEO_PATH = str(dest)
        _status_label.value = f"Saved to: {VIDEO_PATH}"
        print(f"Saved to: {VIDEO_PATH}")
        print("File exists:", os.path.exists(VIDEO_PATH))

    _upload_widget.observe(_on_upload_change, names="value")
    display(_upload_widget, _status_label)
    print("After selecting a file, wait for the 'Saved to...' confirmation, then run the next cell.")


In [ ]:
assert VIDEO_PATH is not None and os.path.exists(VIDEO_PATH), (
    "VIDEO_PATH is not set. Run the upload cell above and select a video file "
    "(in Jupyter: click 'Upload video' and wait for the 'Saved to...' "
    "message before re-running this cell)."
)
print("Video ready:", VIDEO_PATH)


## 4. Scene Calibration

`SceneCalibration` holds the geometric definitions that map pixel coordinates to semantic zones: lane polygons, permitted direction vectors, no-parking regions, and stop-line rectangles.

`make_demo_calibration()` returns a pre-defined calibration matched to a 2560×1440 reference scene. For a new camera viewpoint, replace the polygon coordinates with those obtained from a calibration tool or manual annotation.

`point_in_polygon` wraps `cv2.pointPolygonTest` for use throughout the violation engine.


In [7]:
@dataclass
class SceneCalibration:
    lane_A:             np.ndarray
    lane_B:             np.ndarray
    no_parking_zones:   List[np.ndarray]
    lane_A_direction:   Tuple[float, float]
    lane_B_direction:   Tuple[float, float]
    frame_width:        int = 1280
    frame_height:       int = 720
    lane_A_stop_line:   Optional[np.ndarray] = None
    lane_B_stop_line:   Optional[np.ndarray] = None


def make_demo_calibration(frame_w: int, frame_h: int) -> SceneCalibration:
    lane_A = np.array([
        [193,1439],[1004,1],[1230,1],[2448,1440],[795,1440]
    ], dtype=np.int32)

    lane_B = np.array([
        [967,0],[78,1439],[0,1439],[0,1435],
        [2,409],[777,1],[834,0]
    ], dtype=np.int32)

    parking_zone = np.array([
        [1319,34],[1259,30],[2452,1440],[2553,1440],
        [2556,1129],[2393,947],[2016,611]
    ], dtype=np.int32)

    lane_A_stop_line = np.array([
        [int(frame_w*0.1), int(frame_h*0.8)],
        [int(frame_w*0.4), int(frame_h*0.8)],
        [int(frame_w*0.4), int(frame_h*0.85)],
        [int(frame_w*0.1), int(frame_h*0.85)],
    ], dtype=np.int32)

    lane_B_stop_line = np.array([
        [int(frame_w*0.6), int(frame_h*0.8)],
        [int(frame_w*0.9), int(frame_h*0.8)],
        [int(frame_w*0.9), int(frame_h*0.85)],
        [int(frame_w*0.6), int(frame_h*0.85)],
    ], dtype=np.int32)

    return SceneCalibration(
        lane_A=lane_A, lane_B=lane_B,
        no_parking_zones=[parking_zone],
        lane_A_direction=(0.0, -1.0),
        lane_B_direction=(0.0, 1.0),
        lane_A_stop_line=lane_A_stop_line,
        lane_B_stop_line=lane_B_stop_line,
        frame_width=frame_w,
        frame_height=frame_h,
    )


def point_in_polygon(point: Tuple[int,int], polygon: np.ndarray) -> bool:
    return cv2.pointPolygonTest(polygon, (float(point[0]), float(point[1])), False) >= 0


print("SceneCalibration ready")


SceneCalibration ready


In [8]:
calib = make_demo_calibration(2560, 1440)
print("lane_A shape:", calib.lane_A.shape)
print("lane_B shape:", calib.lane_B.shape)
print("Directions  :", calib.lane_A_direction, calib.lane_B_direction)


lane_A shape: (5, 2)
lane_B shape: (7, 2)
Directions  : (0.0, -1.0) (0.0, 1.0)


## 5. Frame Ingestion and Preprocessing

`FrameIngestion` wraps `cv2.VideoCapture` and exposes a generator that yields sampled frames as `(frame_index, timestamp_sec, raw_frame)` tuples. Frame selection is governed by `FRAME_SAMPLE_RATE`.

`preprocess_frame` offers two modes:
- **Standard mode (`lite=False`):** applies CLAHE contrast enhancement in LAB colour space followed by an unsharp mask. Recommended for recorded video.
- **Lite mode (`lite=True`):** returns the frame unmodified. Suitable for live-feed scenarios where preprocessing latency is a constraint; the model's internal batch normalisation compensates adequately.


In [9]:
def preprocess_frame(frame: np.ndarray, lite: bool = False) -> np.ndarray:
    """
    Apply contrast enhancement to a BGR frame.

    Parameters
    ----------
    frame : np.ndarray
        Input BGR frame from cv2.VideoCapture.
    lite : bool
        If True, return the frame unchanged (live-feed mode).
        If False, apply CLAHE in LAB space followed by an unsharp mask.

    Returns
    -------
    np.ndarray
        Preprocessed BGR frame.
    """
    if lite:
        return frame

    lab   = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_eq  = clahe.apply(l)
    enhanced = cv2.cvtColor(cv2.merge([l_eq, a, b]), cv2.COLOR_LAB2BGR)
    blurred  = cv2.GaussianBlur(enhanced, (0, 0), 3)
    return cv2.addWeighted(enhanced, 1.5, blurred, -0.5, 0)


class FrameIngestion:
    """Decode a video source and emit uniformly sampled, timestamped frames."""

    def __init__(self, source, sample_rate: int = FRAME_SAMPLE_RATE):
        self.cap         = cv2.VideoCapture(source)
        self.sample_rate = sample_rate
        self.fps         = self.cap.get(cv2.CAP_PROP_FPS) or 25.0
        self.frame_idx   = 0

    @property
    def resolution(self) -> Tuple[int, int]:
        w = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        return w, h

    def frames(self):
        """Yield (frame_index, timestamp_sec, raw_frame) for each sampled frame."""
        while self.cap.isOpened():
            ret, frame = self.cap.read()
            if not ret:
                break
            self.frame_idx += 1
            if self.frame_idx % self.sample_rate != 0:
                continue
            yield self.frame_idx, self.frame_idx / self.fps, frame

    def release(self):
        self.cap.release()


print("FrameIngestion and preprocess_frame defined.")


Ingestion + preprocessing ready


## 6. Object Detection

The `Detector` class wraps a YOLOv8 model and applies a two-stage post-processing filter:

1. **Confidence floor:** each COCO class has an independent minimum confidence threshold defined in `CLASS_CONF_FLOORS`. Detections below the floor are discarded.
2. **Shape filter (large vehicles only):** bounding boxes below `CAR_MIN_BOX_AREA_FRAC` or outside the `[CAR_ASPECT_RATIO_MIN, CAR_ASPECT_RATIO_MAX]` range are rejected. This eliminates road-paint markings (arrows, stop-bar text) that the detector frequently mis-classifies as vehicles.

Additional methods:
- `traffic_light_color` — classifies the dominant hue in a cropped traffic-light region.
- `crop_plate_region` — returns the lower-centre crop of a vehicle box for OCR input.


In [10]:
# COCO class ID constants used throughout the pipeline
COCO_VEHICLE_IDS   = {1: "bicycle", 2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}
COCO_PERSON_ID     = 0
COCO_TRAFFIC_LIGHT = 9
TWO_WHEELER_IDS    = {1, 3}
LARGE_VEHICLE_IDS  = {2, 5, 7}


def _box_area_frac(box: np.ndarray, frame_shape) -> float:
    """Return bounding-box area as a fraction of total frame area."""
    x1, y1, x2, y2 = box
    return ((x2 - x1) * (y2 - y1)) / (frame_shape[0] * frame_shape[1])


class Detector:
    def __init__(self, model_name: str = MAIN_MODEL_NAME):
        print(f"Loading model: {model_name} ...")
        self.model = YOLO(model_name)
        self._frame_h = None
        self._frame_w = None
        self._frame_count = 0

    def detect(self, frame: np.ndarray) -> sv.Detections:
        h, w = frame.shape[:2]
        self._frame_h, self._frame_w = h, w
        self._frame_count += 1
        do_debug = (DEBUG_DETECTOR and self._frame_count % DEBUG_EVERY_N_FRAMES == 0)

        results = self.model(frame, conf=0.10, imgsz=YOLO_IMGSZ, verbose=False)[0]
        detections = sv.Detections.from_ultralytics(results)

        if do_debug and detections.class_id is not None:
            for cid, conf in zip(detections.class_id, detections.confidence):
                if int(cid) in LARGE_VEHICLE_IDS:
                    print(f"[DEBUG frame {self._frame_count}] {COCO_VEHICLE_IDS[int(cid)]} conf={conf:.3f}")

        if detections.class_id is not None and len(detections) > 0:
            keep_mask = np.ones(len(detections), dtype=bool)
            for i, (cid, conf, box) in enumerate(zip(detections.class_id, detections.confidence, detections.xyxy)):
                floor = CLASS_CONF_FLOORS.get(int(cid), YOLO_CONF_THRESHOLD)
                if conf < floor:
                    keep_mask[i] = False
                    continue
                # Minimum-area guard for motorcycles
                if int(cid) == 3 and _box_area_frac(box, (h, w)) < MOTO_MIN_BOX_AREA_FRAC:
                    keep_mask[i] = False
                    continue
                # Shape-based false-positive rejection for large vehicles.
                # Road-paint markings fail the area threshold or have extreme aspect ratios.
                if int(cid) in LARGE_VEHICLE_IDS:
                    bw = box[2] - box[0]
                    bh = box[3] - box[1]
                    if _box_area_frac(box, (h, w)) < CAR_MIN_BOX_AREA_FRAC:
                        keep_mask[i] = False
                        continue
                    if bh > 0:
                        aspect = bw / bh
                        if not (CAR_ASPECT_RATIO_MIN <= aspect <= CAR_ASPECT_RATIO_MAX):
                            keep_mask[i] = False
                            continue
            detections = detections[keep_mask]

        return detections

    def traffic_light_color(self, frame: np.ndarray, box: np.ndarray) -> str:
        x1, y1, x2, y2 = map(int, box)
        crop = frame[y1:y2, x1:x2]
        if crop.size == 0: return "unknown"
        hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
        masks = {
            "red": cv2.inRange(hsv,(0,120,120),(10,255,255)) | cv2.inRange(hsv,(160,120,120),(180,255,255)),
            "yellow": cv2.inRange(hsv,(15,120,120),(35,255,255)),
            "green":  cv2.inRange(hsv,(40,80,80),(90,255,255)),
        }
        scores = {c: m.sum() for c, m in masks.items()}
        best   = max(scores, key=scores.get)
        return best if scores[best] > 500 else "unknown"

    def crop_plate_region(self, frame: np.ndarray, box: np.ndarray) -> Optional[np.ndarray]:
        x1, y1, x2, y2 = map(int, box)
        h = y2 - y1
        plate_y1 = y1 + int(h * 0.75)
        cx = (x1 + x2) // 2
        half_w = max(int((x2 - x1) * 0.35), 30)
        crop = frame[plate_y1:y2, max(cx-half_w,0):min(cx+half_w, frame.shape[1])]
        return crop if crop.size > 0 else None


detector = Detector(MAIN_MODEL_NAME)


Loading main model: yolov8n.pt ...


## 7. Helmet Detection

A dedicated binary-classification YOLO model detects helmet presence on two-wheeler riders. The model operates on cropped sub-images centred on the rider's head region rather than the full frame, reducing inference cost and improving accuracy on small targets.

**Head-crop extraction (`crop_rider_region`):** the vehicle bounding box from the main detector is extended upward by `HELMET_PAD_TOP_FRAC × box_height` and laterally by `HELMET_PAD_SIDE_FRAC × box_width`, then clamped to frame boundaries. Only the upper 60% of the extended crop is retained to exclude wheels and the vehicle body.

**Multi-frame voting:** individual per-frame predictions are unreliable due to partial occlusion and pose variation. `TrackState.helmet_decision()` aggregates results over a rolling window of `HELMET_VOTE_WINDOW` frames and applies a majority threshold (`HELMET_VOTE_THRESH`) before issuing a no-helmet violation.


In [11]:
# Clone the helmet detection model weights repository (shallow clone, ~1 commit)
!git clone --depth 1 https://github.com/Juliowiwiwiwi/Bike-Helmet-Detction-Model.git helmet_src -q

helmet_model = YOLO("helmet_src/Weights/best.pt")
print("Helmet model classes:", helmet_model.names)

# Identify class indices corresponding to helmet-present outcomes
HELMET_PRESENT_CLASSES = {
    idx for idx, name in helmet_model.names.items()
    if "Without" not in name and "without" not in name
}
print("Helmet-present class IDs:", HELMET_PRESENT_CLASSES)


Helmet model classes: {0: 'With Helmet', 1: 'Without Helmet'}
Helmet-present class ids: {0}


In [12]:
def crop_rider_region(frame: np.ndarray, vehicle_box: np.ndarray) -> Optional[np.ndarray]:
    """
    Extract the rider head region from a two-wheeler bounding box.

    The YOLO vehicle box encloses the full motorcycle from wheel to handlebar.
    The rider's helmet occupies the upper portion of that box and may extend
    above the top edge when the vehicle is close to the camera.

    The crop is extended upward by HELMET_PAD_TOP_FRAC × box_height and
    laterally by HELMET_PAD_SIDE_FRAC × box_width, clamped to frame boundaries.
    Only the top 60% of the extended region is retained to exclude the vehicle
    body and wheels from the helmet classifier input.

    Parameters
    ----------
    frame : np.ndarray
        Full BGR frame.
    vehicle_box : np.ndarray
        Bounding box [x1, y1, x2, y2] from the main detector.

    Returns
    -------
    np.ndarray or None
        Cropped BGR image, or None if the resulting crop has zero area.
    """
    x1, y1, x2, y2 = map(int, vehicle_box)
    h = y2 - y1
    w = x2 - x1

    pad_top  = int(h * HELMET_PAD_TOP_FRAC)
    pad_side = int(w * HELMET_PAD_SIDE_FRAC)

    crop_y1 = max(0,              y1 - pad_top)
    crop_y2 = min(frame.shape[0], y1 + int(h * 0.60))
    crop_x1 = max(0,              x1 - pad_side)
    crop_x2 = min(frame.shape[1], x2 + pad_side)

    crop = frame[crop_y1:crop_y2, crop_x1:crop_x2]
    return crop if crop.size > 0 else None


## 8. Multi-Object Tracking

### Dual-Tracker Design

Motorcycles and large vehicles require mutually incompatible ByteTrack parameters:
- Motorcycles are frequently detected at low confidence (floor 0.18) and may be occluded for several frames. A high activation threshold or multi-frame confirmation requirement would silently drop valid tracks.
- Cars, buses, and trucks are visually salient; road-paint false positives rarely persist across two consecutive frames, so strict confirmation settings eliminate them without degrading recall.

Two independent `sv.ByteTrack` instances are maintained — `moto_tracker` for persons, bicycles, and motorcycles, and `lv_tracker` for large vehicles — and their outputs are merged into a single `states` dictionary.

### Track ID Collision Avoidance

Each ByteTrack instance maintains its own sequential counter starting from 1. To prevent a motorcycle and a car from sharing the same integer ID, all large-vehicle track IDs are offset by `LV_TRACK_ID_OFFSET` (50,000). Downstream components receive plain integer IDs with no awareness of the offset.

### TrackState

`TrackState` accumulates the full history of a track across frames: bounding boxes, confidence scores, frame indices, timestamps, and bounded deques for helmet-detection and direction votes used in multi-frame decision logic.


In [13]:
@dataclass
class TrackState:
    """Accumulated per-track observation history and derived state."""
    track_id:       int
    class_id:       int
    class_name:     str
    boxes:          List[np.ndarray]  = field(default_factory=list)
    confidences:    List[float]       = field(default_factory=list)
    frame_indices:  List[int]         = field(default_factory=list)
    timestamps:     List[float]       = field(default_factory=list)
    rider_counts:   List[int]         = field(default_factory=list)

    # Bounded deque of per-frame helmet detection outcomes (True = helmet present).
    helmet_seen:    deque = field(
        default_factory=lambda: deque(maxlen=HELMET_VOTE_WINDOW)
    )
    # Bounded deque of per-frame wrong-side direction votes (True = wrong direction).
    direction_votes: deque = field(
        default_factory=lambda: deque(maxlen=WRONG_SIDE_VOTE_WINDOW)
    )

    def latest_box(self) -> Optional[np.ndarray]:
        return self.boxes[-1] if self.boxes else None

    def center(self, box: np.ndarray) -> Tuple[float, float]:
        x1, y1, x2, y2 = box
        return ((x1+x2)/2, (y1+y2)/2)

    def displacement(self) -> float:
        if len(self.boxes) < 2: return 0.0
        c1 = self.center(self.boxes[-2])
        c2 = self.center(self.boxes[-1])
        return float(np.linalg.norm(np.array(c2) - np.array(c1)))

    def velocity_vector(self) -> Tuple[float, float]:
        if len(self.boxes) < 2: return (0.0, 0.0)
        c1 = self.center(self.boxes[-2])
        c2 = self.center(self.boxes[-1])
        return (c2[0]-c1[0], c2[1]-c1[1])

    def helmet_decision(self) -> Optional[bool]:
        """
        Aggregate helmet observations over the rolling vote window.

        Returns True if the fraction of helmet-present frames meets
        HELMET_VOTE_THRESH, False if it does not, or None if fewer than
        half the window has been filled.
        """
        votes = list(self.helmet_seen)
        if len(votes) < max(2, HELMET_VOTE_WINDOW // 2):
            return None
        helmet_fraction = sum(votes) / len(votes)
        return helmet_fraction >= HELMET_VOTE_THRESH

    def wrong_side_decision(self, active_lane_dir: Optional[Tuple[float, float]]) -> Optional[bool]:
        """
        Aggregate direction observations over the rolling vote window.

        A single noisy velocity reading is insufficient to flag wrong-side driving.
        This method applies the same majority-vote logic as helmet_decision().
        """
        if active_lane_dir is None:
            return None
        vx, vy = self.velocity_vector()
        speed  = (vx**2 + vy**2) ** 0.5
        if speed > PIXEL_MOVE_THRESH:
            dot = vx*active_lane_dir[0] + vy*active_lane_dir[1]
            self.direction_votes.append(dot < 0)
        votes = list(self.direction_votes)
        if len(votes) < max(2, WRONG_SIDE_VOTE_WINDOW // 2):
            return None
        return (sum(votes) / len(votes)) >= WRONG_SIDE_VOTE_THRESH

    def stationary_duration(self, fps: float, tolerance: float = 0.85) -> float:
        sample_rate = fps / FRAME_SAMPLE_RATE
        lookback = max(int(PARKING_STILL_SECS * sample_rate) + 5, 2)
        window = self.boxes[-lookback:]
        if len(window) < 2: return 0.0
        still_flags = [
            np.linalg.norm(
                np.array(self.center(window[i])) - np.array(self.center(window[i-1]))
            ) < PIXEL_MOVE_THRESH
            for i in range(1, len(window))
        ]
        return len(still_flags) / sample_rate if sum(still_flags)/len(still_flags) >= tolerance else 0.0


class Tracker:
    """
    Dual ByteTrack wrapper with separate tracker instances per vehicle class group.

    Detections are partitioned into two groups before tracking:
    - TWO_WHEELER_PERSON_IDS (0, 1, 3): routed to moto_tracker with permissive parameters.
    - LARGE_VEH_IDS (2, 5, 7): routed to lv_tracker with strict parameters.

    Track IDs from lv_tracker are offset by LV_TRACK_ID_OFFSET to prevent
    collision with moto_tracker IDs. The merged output dict uses these offset IDs
    transparently throughout downstream pipeline stages.
    """

    TWO_WHEELER_PERSON_IDS = frozenset({0, 1, 3})
    LARGE_VEH_IDS          = frozenset({2, 5, 7})

    def __init__(self, fps: float = 25.0):
        self.moto_tracker = sv.ByteTrack(
            track_activation_threshold = MOTO_TRACK_ACTIVATION_THRESH,
            lost_track_buffer          = MOTO_LOST_TRACK_BUFFER,
            minimum_matching_threshold = 0.80,
            frame_rate                 = int(fps),
            minimum_consecutive_frames = MOTO_MIN_CONSECUTIVE_FRAMES,
        )
        self.lv_tracker = sv.ByteTrack(
            track_activation_threshold = LV_TRACK_ACTIVATION_THRESH,
            lost_track_buffer          = LV_LOST_TRACK_BUFFER,
            minimum_matching_threshold = 0.80,
            frame_rate                 = int(fps),
            minimum_consecutive_frames = LV_MIN_CONSECUTIVE_FRAMES,
        )
        self.states: Dict[int, TrackState] = {}
        self.fps = fps

    def update(self, detections: sv.Detections,
               frame_idx: int, timestamp: float) -> Dict[int, TrackState]:

        if detections.class_id is None or len(detections) == 0:
            self._prune(frame_idx)
            return self.states

        moto_mask = np.isin(detections.class_id, list(self.TWO_WHEELER_PERSON_IDS))
        lv_mask   = np.isin(detections.class_id, list(self.LARGE_VEH_IDS))

        moto_dets = detections[moto_mask]
        lv_dets   = detections[lv_mask]

        groups = [
            (self.moto_tracker, moto_dets, 0),
            (self.lv_tracker,   lv_dets,   LV_TRACK_ID_OFFSET),
        ]
        for tracker_inst, dets, id_offset in groups:
            tracked = tracker_inst.update_with_detections(dets)
            if tracked.tracker_id is None or len(tracked) == 0:
                continue
            for i, raw_tid in enumerate(tracked.tracker_id):
                tid  = int(raw_tid) + id_offset
                cid  = int(tracked.class_id[i]) if tracked.class_id is not None else -1
                name = COCO_VEHICLE_IDS.get(cid, "person" if cid == COCO_PERSON_ID else "other")
                box  = tracked.xyxy[i]
                conf = float(tracked.confidence[i]) if tracked.confidence is not None else 0.0

                if tid not in self.states:
                    self.states[tid] = TrackState(track_id=tid, class_id=cid, class_name=name)
                s = self.states[tid]
                s.boxes.append(box)
                s.confidences.append(conf)
                s.frame_indices.append(frame_idx)
                s.timestamps.append(timestamp)

        self._prune(frame_idx)
        return self.states

    def _prune(self, frame_idx: int) -> None:
        """Remove track states that have not been updated within TRACK_MAX_AGE_FRAMES."""
        stale_ids = [
            tid for tid, s in self.states.items()
            if s.frame_indices and
               (frame_idx - s.frame_indices[-1]) > TRACK_MAX_AGE_FRAMES
        ]
        for tid in stale_ids:
            del self.states[tid]


print("TrackState and Tracker defined.")


TrackState + Tracker ready


## 9. Violation Detection Engine

`ViolationEngine` evaluates each confirmed track against five geometric and temporal rules:

| Rule | Condition |
|------|-----------|
| No-helmet | Two-wheeler track with `helmet_decision()` returning False after `MIN_CONFIRM_FRAMES` |
| Triple riding | Two-wheeler with `rider_count > 2` in any of the last 5 frames |
| Red-light crossing | Vehicle footprint intersects the active stop-line polygon while `light_color == "red"` |
| Illegal parking | Non-person vehicle stationary for ≥ `PARKING_STILL_SECS` seconds inside a no-parking zone |
| Wrong-side driving | `wrong_side_decision()` returns True after the direction vote window fills |

Each violation type is flagged at most once per track (tracked via `_flagged`). A `ViolationEvent` is returned when one or more rules fire.


In [14]:
@dataclass
class ViolationEvent:
    track_id:       int
    class_name:     str
    violations:     List[str]
    frame_idx:      int
    timestamp:      float
    box:            np.ndarray
    confidence:     float
    plate_text:     str = ""
    evidence_frame: Optional[np.ndarray] = field(default=None, repr=False)


class ViolationEngine:
    """Evaluate per-track state against calibration geometry to produce violation events."""

    def __init__(self, calibration: SceneCalibration, fps: float = 25.0):
        self.calib = calibration
        self.fps   = fps
        self._flagged: Dict[int, set] = defaultdict(set)

    def evaluate(
        self,
        state:       TrackState,
        frame_idx:   int,
        timestamp:   float,
        light_color: str = "unknown",
    ) -> Optional[ViolationEvent]:

        box = state.latest_box()
        if box is None:
            return None

        x1, y1, x2, y2 = map(int, box)
        leading_edge = ((x1+x2)//2, y2)
        violations   = []

        # Determine which lane the vehicle occupies and resolve the corresponding
        # direction vector and stop-line polygon for that lane.
        in_lane_A = point_in_polygon(leading_edge, self.calib.lane_A)
        in_lane_B = point_in_polygon(leading_edge, self.calib.lane_B)
        active_lane_dir  = self.calib.lane_A_direction if in_lane_A else (
                           self.calib.lane_B_direction if in_lane_B else None)
        active_stop_line = self.calib.lane_A_stop_line if in_lane_A else (
                           self.calib.lane_B_stop_line if in_lane_B else None)

        # Rule 1: No-helmet — multi-frame vote must reach a decision before flagging.
        if (ENABLE_HELMET
                and state.class_id in TWO_WHEELER_IDS
                and len(state.boxes) >= MIN_CONFIRM_FRAMES):
            decision = state.helmet_decision()
            if decision is False:
                if "no_helmet" not in self._flagged[state.track_id]:
                    violations.append("no_helmet")

        # Rule 2: Triple riding
        if (ENABLE_TRIPLE_RIDING
                and state.class_id in TWO_WHEELER_IDS
                and state.rider_counts
                and max(state.rider_counts[-5:] or [0]) > 2):
            if "triple_riding" not in self._flagged[state.track_id]:
                violations.append("triple_riding")

        # Rule 3: Red-light / stop-line crossing
        if ENABLE_STOP_LINE and light_color == "red" and active_stop_line is not None:
            if point_in_polygon(leading_edge, active_stop_line):
                if "red_light" not in self._flagged[state.track_id]:
                    violations.append("red_light")

        # Rule 4: Illegal parking (persons excluded)
        if (ENABLE_NO_PARKING
                and state.class_id != COCO_PERSON_ID
                and self.calib.no_parking_zones):
            still_secs = state.stationary_duration(self.fps)
            if still_secs >= PARKING_STILL_SECS:
                for zone in self.calib.no_parking_zones:
                    if point_in_polygon(leading_edge, zone):
                        if "illegal_parking" not in self._flagged[state.track_id]:
                            violations.append("illegal_parking")
                        break

        # Rule 5: Wrong-side driving (persons excluded; requires valid lane assignment)
        if ENABLE_WRONG_SIDE and state.class_id != COCO_PERSON_ID:
            decision = state.wrong_side_decision(active_lane_dir)
            if decision is True:
                if "wrong_side" not in self._flagged[state.track_id]:
                    violations.append("wrong_side")

        if not violations:
            return None

        self._flagged[state.track_id].update(violations)
        return ViolationEvent(
            track_id   = state.track_id,
            class_name = state.class_name,
            violations = violations,
            frame_idx  = frame_idx,
            timestamp  = timestamp,
            box        = box,
            confidence = state.confidences[-1] if state.confidences else 0.0,
        )


print("ViolationEngine defined.")


ViolationEngine ready


## 10. License Plate OCR

`PlateOCR` wraps EasyOCR with lazy initialisation (the reader is loaded on first use) and CPU-only inference to reserve GPU resources for the YOLO detection models.

The raw OCR output is matched against a regular expression for Indian number plates (`[A-Z]{2}\d{1,2}[A-Z]{1,2}\d{4}`). If the pattern is found, the matched string is returned stripped of spaces. Otherwise, the first 12 characters of the raw output are returned.

Input crops narrower than 120 px are upscaled before OCR to improve character recognition accuracy.


In [15]:
import re

PLATE_PATTERN = re.compile(r"[A-Z]{2}\s?\d{1,2}\s?[A-Z]{1,2}\s?\d{4}")


class PlateOCR:
    """EasyOCR wrapper with lazy initialisation and Indian plate-format parsing."""

    def __init__(self):
        self._reader = None

    @property
    def reader(self):
        if self._reader is None:
            print("Initialising EasyOCR reader...")
            self._reader = easyocr.Reader(["en"], gpu=False)
            print("OCR reader ready.")
        return self._reader

    def read_plate(self, crop: np.ndarray) -> str:
        if crop is None or crop.size == 0:
            return ""
        h, w = crop.shape[:2]
        if w < 120:
            crop = cv2.resize(crop, None,
                              fx=120/w, fy=120/w, interpolation=cv2.INTER_CUBIC)
        results = self.reader.readtext(crop, detail=0, paragraph=False)
        raw     = " ".join(results).upper().strip()
        match   = PLATE_PATTERN.search(raw)
        return match.group(0).replace(" ", "") if match else raw[:12]


plate_ocr = PlateOCR()
print("PlateOCR defined (reader initialised on first violation).")


PlateOCR ready (reader loads on first violation)


## 11. Rendering and Evidence Storage

**`draw_dashboard_overlay`** renders an always-on annotation layer: lane polygon outlines, no-parking zone boundaries, stop-line rectangles, and bounding boxes with track IDs for all confirmed tracks. Tracks not updated within `TRACK_DISPLAY_AGE` frames are suppressed to prevent ghost overlays from stale associations.

**`render_evidence`** produces a single annotated frame for a violation event: semi-transparent zone fills, the vehicle bounding box in the violation-type colour, and text annotations (track ID, class, violation labels, plate text, timestamp, confidence).

**`EvidenceStore`** persists each violation as a JPEG evidence frame under `violation_output/frames/` and appends a JSON record to `violations.jsonl`. The log is flushed after each write to ensure durability.


In [16]:
VIOLATION_COLORS = {
    "no_helmet":       (0,   0,   255),
    "triple_riding":   (0,   165, 255),
    "red_light":       (0,   0,   200),
    "illegal_parking": (255, 0,   255),
    "wrong_side":      (255, 255, 0),
}


def draw_dashboard_overlay(frame: np.ndarray,
                            states: Dict[int, "TrackState"],
                            calib: SceneCalibration,
                            current_frame_idx: int = 0) -> np.ndarray:
    """
    Render lane geometry, zone boundaries, and confirmed track boxes onto a frame.

    Tracks not updated within TRACK_DISPLAY_AGE frames are excluded to prevent
    stale bounding boxes from persisting after a vehicle has left the scene.
    """
    out = frame.copy()
    cv2.polylines(out, [calib.lane_A], True, (0,255,0), 3)
    cv2.polylines(out, [calib.lane_B], True, (255,0,0), 3)
    if ENABLE_NO_PARKING:
        for zone in calib.no_parking_zones:
            cv2.polylines(out, [zone], True, (0,0,255), 3)
    if ENABLE_STOP_LINE:
        if calib.lane_A_stop_line is not None:
            cv2.polylines(out, [calib.lane_A_stop_line], True, (0,200,200), 3)
        if calib.lane_B_stop_line is not None:
            cv2.polylines(out, [calib.lane_B_stop_line], True, (0,200,200), 3)
    for tid, state in states.items():
        if len(state.boxes) < MIN_CONFIRM_FRAMES:
            continue
        if state.frame_indices and                 (current_frame_idx - state.frame_indices[-1]) > TRACK_DISPLAY_AGE:
            continue
        box = state.latest_box()
        if box is None:
            continue
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(out, (x1,y1), (x2,y2), (0,255,255), 2)
        cv2.putText(out, f"ID {tid} {state.class_name}", (x1, max(0,y1-8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,255), 2)
    return out


def render_evidence(frame: np.ndarray,
                    event: ViolationEvent,
                    calib: SceneCalibration) -> np.ndarray:
    out = frame.copy()
    overlay = out.copy()
    cv2.polylines(overlay, [calib.lane_A], True, (0,255,0), 4)
    cv2.polylines(overlay, [calib.lane_B], True, (255,0,0), 4)
    for zone in calib.no_parking_zones:
        cv2.fillPoly(overlay, [zone], (0,0,180))
    cv2.addWeighted(overlay, 0.25, out, 0.75, 0, out)

    x1, y1, x2, y2 = map(int, event.box)
    color = VIOLATION_COLORS.get(event.violations[0], (0,0,255))
    cv2.rectangle(out, (x1,y1), (x2,y2), color, 2)

    for j, line in enumerate([
        f"ID:{event.track_id}  {event.class_name}",
        " | ".join(event.violations),
        f"Plate: {event.plate_text or 'N/A'}",
        f"T={event.timestamp:.1f}s  conf={event.confidence:.2f}",
    ]):
        cv2.putText(out, line, (x1, y1 - 10 - j*18),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)

    cv2.putText(out, f"Frame {event.frame_idx}",
                (10, out.shape[0]-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200,200,200), 1)
    return out


class EvidenceStore:
    def __init__(self, output_dir: Path = OUTPUT_DIR):
        self.dir      = output_dir
        self.log_path = output_dir / "violations.jsonl"
        self._log     = open(self.log_path, "w")
        self.count    = 0

    def save(self, event: ViolationEvent, annotated_frame: np.ndarray) -> dict:
        self.count += 1
        fname = f"viol_{self.count:04d}_id{event.track_id}_f{event.frame_idx}.jpg"
        cv2.imwrite(str(self.dir / "frames" / fname), annotated_frame)
        record = {
            "id":         self.count,
            "track_id":   event.track_id,
            "class":      event.class_name,
            "violations": event.violations,
            "timestamp":  round(event.timestamp, 2),
            "frame_idx":  event.frame_idx,
            "plate":      event.plate_text,
            "confidence": round(event.confidence, 3),
            "evidence":   fname,
        }
        self._log.write(json.dumps(record, default=str) + "\n")
        self._log.flush()
        return record

    def close(self):
        self._log.close()


print("Rendering functions and EvidenceStore defined.")


Evidence generation ready


## 12. Pipeline Execution

`run_pipeline` orchestrates the full processing loop over the input video. For each sampled frame, the function:

1. Preprocesses the frame according to `LIVE_FEED_MODE`.
2. Runs the main detector and extracts the current traffic-light state.
3. Routes vehicle detections to the tracker.
4. Renders the dashboard overlay onto the output frame.
5. Evaluates each confirmed track for violations; on a hit, runs OCR on the plate region, saves evidence, and annotates the output frame.
6. Writes the output frame to the annotated video file.

All output artefacts are written to `OUTPUT_DIR`:
- `annotated_output_fixed.mp4` — full annotated video
- `frames/` — per-violation evidence images
- `violations.jsonl` — newline-delimited JSON violation log


In [17]:
def run_pipeline(source=VIDEO_SOURCE, max_frames: int = 500, show_preview: bool = False):
    ingestion = FrameIngestion(source)
    w, h = ingestion.resolution
    fps = ingestion.fps
    calib = make_demo_calibration(w, h)
    tracker = Tracker(fps=fps)
    engine = ViolationEngine(calib, fps=fps)
    store = EvidenceStore()
    video_path = OUTPUT_DIR / "annotated_output_fixed.mp4"
    writer = cv2.VideoWriter(str(video_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    records = []; processed = 0; t_start = time.time(); light_color = "unknown"

    for f_idx, ts, raw in ingestion.frames():
        if max_frames and processed >= max_frames: break
        processed += 1
        clean = preprocess_frame(raw, lite=LIVE_FEED_MODE)
        frame_to_output = clean.copy()

        detections = detector.detect(clean)

        if ENABLE_STOP_LINE and detections.class_id is not None:
            tl_mask = detections.class_id == COCO_TRAFFIC_LIGHT
            if tl_mask.any():
                light_color = detector.traffic_light_color(clean, detections.xyxy[tl_mask][0])

        if detections.class_id is not None:
            keep_ids = set(COCO_VEHICLE_IDS) | {COCO_PERSON_ID}
            vehicle_dets = detections[np.isin(detections.class_id, list(keep_ids))]
            states = tracker.update(vehicle_dets, f_idx, ts)

            frame_to_output = draw_dashboard_overlay(clean.copy(), states, calib,
                                                     current_frame_idx=f_idx)

            for tid, state in states.items():
                if state.class_id in TWO_WHEELER_IDS and ENABLE_HELMET:
                    crop = crop_rider_region(raw, state.latest_box())
                    if crop is not None:
                        hres = helmet_model(crop, conf=HELMET_CONF, verbose=False)[0]
                        state.helmet_seen.append(len(hres.boxes) > 0 and int(hres.boxes.cls[0]) in HELMET_PRESENT_CLASSES)

                event = engine.evaluate(state, f_idx, ts, light_color)
                if event:
                    event.plate_text = plate_ocr.read_plate(detector.crop_plate_region(clean, event.box))
                    annotated_event_frame = render_evidence(frame_to_output, event, calib)
                    records.append(store.save(event, annotated_event_frame))
                    frame_to_output = annotated_event_frame

        writer.write(frame_to_output)

    ingestion.release(); store.close(); writer.release()
    print(f"Processing complete. Total violations detected: {len(records)}")
    return records


records = run_pipeline(source=VIDEO_PATH, max_frames=2000)


Initialising EasyOCR (first call only) ...
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% CompleteOCR ready
Done. Violations: 5


## 13. Output Export

The pipeline writes all artefacts to `violation_output/` on disk. This section packages those artefacts into a single zip archive and provides a download mechanism appropriate to the runtime environment.

The zip archive contains:
- `annotated_output_fixed.mp4` — full annotated video
- `frames/` — per-violation JPEG evidence images
- `violations.json` — a JSON array version of the JSONL log, generated here for compatibility with external tools

In Google Colab, `files.download()` triggers an immediate browser download. In a local Jupyter environment, `IPython.display.FileLink` renders an inline download link.


In [ ]:
import json, shutil
from pathlib import Path

# Convert the internal JSONL log to a proper JSON array for distribution
_jsonl_path = OUTPUT_DIR / "violations.jsonl"
_json_path  = OUTPUT_DIR / "violations.json"

if _jsonl_path.exists():
    with open(_jsonl_path) as f:
        _violation_records = [json.loads(line) for line in f if line.strip()]
    with open(_json_path, "w") as f:
        json.dump(_violation_records, f, indent=2, default=str)
    print(f"Wrote {_json_path} ({len(_violation_records)} records)")
else:
    print(f"{_jsonl_path} not found. Run the pipeline cell first.")

# Package all output artefacts into a single archive
_zip_base = "emerald_outputs"
_zip_path = Path(shutil.make_archive(_zip_base, "zip", root_dir=OUTPUT_DIR))
print(f"Archive created: {_zip_path.resolve()}")

# Deliver the archive to the user
if _in_colab():
    from google.colab import files as _colab_files
    print("Initiating browser download...")
    _colab_files.download(str(_zip_path))
else:
    from IPython.display import FileLink, display
    print("Download archive:")
    display(FileLink(str(_zip_path)))

    _video_out = OUTPUT_DIR / "annotated_output_fixed.mp4"
    if _video_out.exists():
        print("\nAnnotated video:")
        display(FileLink(str(_video_out)))
    if _json_path.exists():
        print("\nViolation log (JSON):")
        display(FileLink(str(_json_path)))


In [18]:
from collections import Counter


def load_violation_log(log_path: Path = OUTPUT_DIR / "violations.jsonl"):
    rows = []
    with open(log_path) as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def report(rows):
    if not rows:
        print("No violations recorded.")
        return
    all_viols = [v for r in rows for v in r["violations"]]
    counts    = Counter(all_viols)
    print("\n══ VIOLATION SUMMARY ════════════════════")
    for vtype, n in counts.most_common():
        print(f"  {vtype:<22} {n:>4}")
    print(f"  {'TOTAL':<22} {len(rows):>4}")
    plates = [r["plate"] for r in rows if r["plate"]]
    print(f"\nUnique plates detected: {len(set(plates))}")
    if plates:
        print(f"  e.g. {', '.join(sorted(set(plates))[:5])}")


if 'records' in locals() and records:
    print("\n══ VIOLATION RECORDS ════════════════════")
    for record in records:
        print(record)
elif 'records' in locals() and not records:
    print("No violations recorded.")
else:
    print("Run the pipeline cell before executing this cell.")



══ VIOLATION RECORDS ════════════════════
{'id': 1, 'track_id': 10, 'class': 'motorcycle', 'violations': ['no_helmet'], 'timestamp': 18.16, 'frame_idx': 454, 'plate': '', 'confidence': 0.67, 'evidence': 'viol_0001_id10_f454.jpg'}
{'id': 2, 'track_id': 50017, 'class': 'bus', 'violations': ['illegal_parking'], 'timestamp': 21.44, 'frame_idx': 536, 'plate': 'FON', 'confidence': 0.917, 'evidence': 'viol_0002_id50017_f536.jpg'}
{'id': 3, 'track_id': 50025, 'class': 'car', 'violations': ['illegal_parking'], 'timestamp': 29.12, 'frame_idx': 728, 'plate': '', 'confidence': 0.594, 'evidence': 'viol_0003_id50025_f728.jpg'}
{'id': 4, 'track_id': 50029, 'class': 'truck', 'violations': ['illegal_parking'], 'timestamp': 33.52, 'frame_idx': 838, 'plate': '', 'confidence': 0.923, 'evidence': 'viol_0004_id50029_f838.jpg'}
{'id': 5, 'track_id': 50027, 'class': 'car', 'violations': ['illegal_parking'], 'timestamp': 38.88, 'frame_idx': 972, 'plate': '', 'confidence': 0.77, 'evidence': 'viol_0005_id50027_

In [19]:
from sklearn.metrics import classification_report


def evaluate_performance(ground_truth: List[dict], predictions: List[dict],
                          violation_type: str = "no_helmet"):
    """
    Compute precision, recall, and F1 for a single violation type.

    Parameters
    ----------
    ground_truth : list of dict
        Manually verified violations. Each entry must contain 'frame_idx',
        'track_id', and 'violation' keys.
    predictions : list of dict
        Pipeline output records (the 'records' list from run_pipeline).
    violation_type : str
        The violation category to evaluate.
    """
    gt_keys   = {(r["frame_idx"], r["track_id"]) for r in ground_truth
                 if r["violation"] == violation_type}
    pred_keys = {(r["frame_idx"], r["track_id"]) for r in predictions
                 if violation_type in r["violations"]}
    all_keys  = gt_keys | pred_keys
    y_true    = [1 if k in gt_keys   else 0 for k in all_keys]
    y_pred    = [1 if k in pred_keys else 0 for k in all_keys]
    print(f"\n── {violation_type.upper()} ──")
    print(classification_report(y_true, y_pred,
                                 target_names=["clean", violation_type],
                                 zero_division=0))


def benchmark_fps(source, n_frames: int = 100):
    """Measure end-to-end throughput (preprocessing + detection) over n_frames."""
    ing   = FrameIngestion(source)
    count = 0
    t0    = time.time()
    for _, _, raw in ing.frames():
        clean = preprocess_frame(raw, lite=LIVE_FEED_MODE)
        _     = detector.detect(clean)
        count += 1
        if count >= n_frames:
            break
    ing.release()
    elapsed = time.time() - t0
    print(f"{count} frames in {elapsed:.2f}s → {count/elapsed:.1f} fps")


print("evaluate_performance() and benchmark_fps() defined.")


Evaluation helpers defined.
Call benchmark_fps(VIDEO_PATH) to measure throughput.
Call evaluate_performance(ground_truth, records) once you have labels.


## 14. Analytics and Reporting

The cells below read from the `violations.jsonl` log produced by `EvidenceStore` and do not modify pipeline state. They may be executed independently after `run_pipeline` has completed.

- **ViolationAnalytics** — loads the log into a pandas DataFrame and provides aggregate statistics, parameterised search, trend visualisation, and HTML report export.
- **Computational efficiency benchmark** — re-runs detection and tracking on fresh object instances to produce per-stage timing and GPU memory usage.
- **Classification metrics** — compares pipeline predictions against manually verified ground-truth labels.
- **Plate OCR accuracy** — computes exact-match rate against human-verified plate strings.
- **Detection mAP** — computes mAP@50 and mAP@50–95 using `supervision.metrics.MeanAveragePrecision` against hand-labelled bounding boxes.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


class ViolationAnalytics:
    """
    Load violations.jsonl into a pandas DataFrame and expose summary, search,
    visualisation, and HTML export methods.
    """

    def __init__(self, log_path: Path = OUTPUT_DIR / "violations.jsonl"):
        rows = [json.loads(l) for l in open(log_path)]
        self.df = pd.json_normalize(rows)
        if not self.df.empty:
            # Convert violation lists to tuples for hashable grouping operations
            self.df["violations"] = self.df["violations"].apply(tuple)

    def stats(self) -> dict:
        df = self.df
        if df.empty:
            return {"total_violations": 0}
        exploded = df.explode("violations")
        return {
            "total_violations":   len(df),
            "by_type":            exploded["violations"].value_counts().to_dict(),
            "by_vehicle_class":   df["class"].value_counts().to_dict(),
            "avg_confidence":     round(df["confidence"].mean(), 3),
            "plates_read":        int((df["plate"] != "").sum()),
            "busiest_10s_window": int(df["timestamp"].round(-1).value_counts().idxmax()),
        }

    def search(self, violation_type=None, vehicle_class=None,
               min_confidence=None, plate_contains=None,
               start_time=None, end_time=None) -> pd.DataFrame:
        df = self.df.copy()
        if violation_type:   df = df[df["violations"].apply(lambda v: violation_type in v)]
        if vehicle_class:    df = df[df["class"] == vehicle_class]
        if min_confidence:   df = df[df["confidence"] >= min_confidence]
        if plate_contains:   df = df[df["plate"].str.contains(plate_contains, na=False)]
        if start_time:       df = df[df["timestamp"] >= start_time]
        if end_time:         df = df[df["timestamp"] <= end_time]
        return df.sort_values("timestamp")

    def plot_trends(self):
        if self.df.empty:
            print("No violations recorded.")
            return
        exploded = self.df.explode("violations")
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        exploded["violations"].value_counts().plot(kind="bar", ax=axes[0], color="#2cc4f0")
        axes[0].set_title("Violations by Type"); axes[0].set_ylabel("Count")
        self.df.set_index(pd.to_timedelta(self.df["timestamp"], unit="s")) \
               .resample("10s").size().plot(ax=axes[1], color="#ff7a7a")
        axes[1].set_title("Violations Over Video Time (10s bins)")
        plt.tight_layout(); plt.savefig(OUTPUT_DIR / "trends.png", dpi=120)
        plt.show()

    def export_report(self, path: Path = OUTPUT_DIR / "summary_report.html"):
        s = self.stats()
        html = f"""<h2>EMERALD Violation Summary</h2>
        <p>Total violations: {s.get('total_violations', 0)}</p>
        <p>Mean detection confidence: {s.get('avg_confidence', '-')}</p>
        <h3>By Type</h3>{pd.Series(s.get('by_type', {})).to_frame('count').to_html()}
        <h3>By Vehicle Class</h3>{pd.Series(s.get('by_vehicle_class', {})).to_frame('count').to_html()}
        <h3>Full Record</h3>{self.df.to_html(index=False)}"""
        path.write_text(html)
        print(f"Report written to {path}")


analytics = ViolationAnalytics()
print(analytics.stats())
analytics.plot_trends()
analytics.export_report()

print(analytics.search(violation_type="no_helmet", min_confidence=0.5))


In [ ]:
import torch


def benchmark_pipeline(source, n_frames: int = 200, required_fps: float = 15.0):
    """
    Measure per-stage latency and end-to-end throughput over n_frames.

    Uses independent FrameIngestion and Tracker instances to avoid
    interfering with the state produced by run_pipeline.

    Parameters
    ----------
    source : str or int
        Video path or camera index.
    n_frames : int
        Number of sampled frames to process.
    required_fps : float
        Target frame rate for scalability estimation.
    """
    ing = FrameIngestion(source)
    tracker_bench = Tracker(fps=ing.fps)
    stage_times = defaultdict(float)
    count = 0
    for f_idx, ts, raw in ing.frames():
        t0 = time.time(); clean = preprocess_frame(raw, lite=LIVE_FEED_MODE); t1 = time.time()
        dets = detector.detect(clean);                                        t2 = time.time()
        _ = tracker_bench.update(dets, f_idx, ts);                            t3 = time.time()
        stage_times["preprocess"] += t1 - t0
        stage_times["detect"]     += t2 - t1
        stage_times["track"]      += t3 - t2
        count += 1
        if count >= n_frames:
            break
    ing.release()
    total = sum(stage_times.values())
    fps = count / total if total > 0 else 0
    print(f"{count} frames, {fps:.1f} fps end-to-end (preprocess + detect + track)")
    for k, v in stage_times.items():
        print(f"  {k}: {v/count*1000:.1f} ms/frame")
    if torch.cuda.is_available():
        print(f"Peak GPU memory: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
    if fps > 0:
        print(f"Estimated concurrent camera streams at {required_fps} fps/stream: {fps/required_fps:.1f}")
    return fps


measured_fps = benchmark_pipeline(VIDEO_PATH)


In [ ]:
# Violation classification evaluation — Precision / Recall / F1
#
# Populate ground_truth with violations verified by manual review of
# annotated_output_fixed.mp4. Do not derive ground_truth from `records`,
# as this would compare the model against itself.
#
# Required keys per entry: frame_idx (int), track_id (int), violation (str).
# Leave the list empty to skip this cell without error.

ground_truth = [
    # Example entry (replace with manually verified observations):
    # {"frame_idx": 454, "track_id": 10, "violation": "no_helmet"},
]

if not ground_truth:
    print("ground_truth is empty. Populate it with manually verified violation "
          "observations from annotated_output_fixed.mp4, then re-run this cell.")
else:
    for vtype in ["no_helmet", "triple_riding", "red_light", "illegal_parking", "wrong_side"]:
        evaluate_performance(ground_truth, records, violation_type=vtype)


In [ ]:
# License plate OCR accuracy — exact-match rate
#
# For each violation with a non-empty plate reading, verify the prediction
# against the actual plate visible in the corresponding evidence frame
# under violation_output/frames/. Leave the list empty to skip.
#
# Each entry: (predicted_string, actual_string_from_evidence_frame).

plate_checks = [
    # Example entry:
    # ("FON", "KA05MN1234"),
]

if not plate_checks:
    print("plate_checks is empty. Populate it with (predicted, actual) pairs "
          "verified against evidence frames, then re-run this cell.")
else:
    exact_match_rate = sum(p == a for p, a in plate_checks) / len(plate_checks)
    print(f"Plate OCR exact-match accuracy: {exact_match_rate:.1%} (n={len(plate_checks)})")


In [ ]:
# Detection mAP evaluation (optional)
#
# Requires hand-annotated bounding boxes for a sample of frames (30–50 frames
# is typically sufficient), labelled using a tool such as CVAT or Roboflow
# and loaded as sv.Detections objects.
#
# If manual annotation is not feasible, the published COCO mAP for the
# pretrained YOLOv8n backbone (from Ultralytics documentation) may be cited
# in lieu of recomputing it, provided this is stated explicitly.
#
# This cell only defines run_map_eval; it is not executed automatically.

from supervision.metrics.mean_average_precision import MeanAveragePrecision


def run_map_eval(ground_truth_boxes: dict, cached_predictions: dict):
    """
    Compute mAP@50 and mAP@50-95 over a set of annotated frames.

    Parameters
    ----------
    ground_truth_boxes : dict
        {frame_idx: sv.Detections} — manual annotations.
    cached_predictions : dict
        {frame_idx: sv.Detections} — detector output on the same frames.
    """
    map_metric = MeanAveragePrecision()
    for frame_idx, gt_dets in ground_truth_boxes.items():
        pred_dets = cached_predictions[frame_idx]
        map_metric.update(pred_dets, gt_dets)
    result = map_metric.compute()
    print(f"mAP@50: {result.map50:.3f}   mAP@50-95: {result.map50_95:.3f}")
    return result


print("run_map_eval() defined.")
